In [10]:
import pandas as pd
import requests

API_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

limit = 50000
offset = 0
all_data = []

while True:

    params = {
        "$limit": limit,
        "$offset": offset
    }

    response = requests.get(API_URL, params=params)

    if response.status_code != 200:
        print("API Error")
        break

    data = response.json()

    if len(data) == 0:
        break

    all_data.extend(data)

    print(f"Downloaded {len(all_data)} records")

    offset += limit

    if len(all_data) >= 100000:
        break

df = pd.DataFrame(all_data)

print(df.shape)
import os

os.makedirs("data/raw", exist_ok=True)

df.to_csv("data/raw/nyc311_raw.csv", index=False)

print("Data Saved Successfully")

Downloaded 50000 records
Downloaded 100000 records
(100000, 48)
Data Saved Successfully


In [11]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'data', 'sample_data']


In [12]:
print(os.listdir('/content/data/raw'))

['nyc311_raw.csv']


In [13]:
import os

print(os.listdir('/content/data'))

['raw']


In [14]:
import pandas as pd

df = pd.read_csv('/content/data/raw/nyc311_raw.csv')

print(df.shape)

/tmp/ipykernel_1438/196870278.py:3: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/data/raw/nyc311_raw.csv')


(100000, 48)


In [15]:
print(df.columns.tolist())

['unique_key', 'created_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'status', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'latitude', 'longitude', 'location', ':@computed_region_f5dn_yrer', ':@computed_region_yeji_bk3q', ':@computed_region_sbqj_enih', ':@computed_region_92fq_4b7q', 'descriptor_2', 'bridge_highway_name', 'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment', 'vehicle_type', 'closed_date', 'resolution_description', 'resolution_action_updated_date', 'facility_type', 'due_date', 'taxi_pick_up_location', 'taxi_company_borough']


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 48 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   unique_key                      100000 non-null  int64  
 1   created_date                    100000 non-null  object 
 2   agency                          100000 non-null  object 
 3   agency_name                     100000 non-null  object 
 4   complaint_type                  100000 non-null  object 
 5   descriptor                      98216 non-null   object 
 6   location_type                   93148 non-null   object 
 7   incident_zip                    99207 non-null   float64
 8   incident_address                97807 non-null   object 
 9   street_name                     97806 non-null   object 
 10  cross_street_1                  71719 non-null   object 
 11  cross_street_2                  71757 non-null   object 
 12  intersection_stre

In [17]:
df.isnull().sum()

,0
unique_key,0
created_date,0
agency,0
agency_name,0
complaint_type,0
descriptor,1784
location_type,6852
incident_zip,793
incident_address,2193
street_name,2194


In [18]:
print("Duplicate rows:", df.duplicated().sum())

print("Duplicate unique_key:",
      df['unique_key'].duplicated().sum())

Duplicate rows: 0
Duplicate unique_key: 0


In [19]:
required_cols = [
    'unique_key',
    'created_date',
    'closed_date',
    'agency',
    'complaint_type',
    'descriptor',
    'borough',
    'status',
    'latitude',
    'longitude'
]

df = df[required_cols]

print(df.shape)

(100000, 10)


Convert date columns

In [20]:
df['created_date'] = pd.to_datetime(
    df['created_date'],
    errors='coerce'
)

df['closed_date'] = pd.to_datetime(
    df['closed_date'],
    errors='coerce'
)

Create derived columns

In [21]:
df['year'] = df['created_date'].dt.year
df['month'] = df['created_date'].dt.month
df['weekday'] = df['created_date'].dt.day_name()
df['hour'] = df['created_date'].dt.hour

df['is_closed'] = df['closed_date'].notna().astype(int)

df['resolution_hours'] = (
    df['closed_date'] - df['created_date']
).dt.total_seconds() / 3600

In [22]:
print(df.columns.tolist())
df.info()
df.isnull().sum()
print(df['unique_key'].duplicated().sum())

['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'borough', 'status', 'latitude', 'longitude', 'year', 'month', 'weekday', 'hour', 'is_closed', 'resolution_hours']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   unique_key        100000 non-null  int64         
 1   created_date      100000 non-null  datetime64[ns]
 2   closed_date       73703 non-null   datetime64[ns]
 3   agency            100000 non-null  object        
 4   complaint_type    100000 non-null  object        
 5   descriptor        98216 non-null   object        
 6   borough           100000 non-null  object        
 7   status            100000 non-null  object        
 8   latitude          98504 non-null   float64       
 9   longitude         98504 non-null   float64       
 10  year              1000

In [23]:
df.to_csv(
    "/content/nyc311_cleaned.csv",
    index=False
)

print("Final cleaned dataset saved")

Final cleaned dataset saved


1.Total Distinct Request Volume

In [ ]:
total_requests = df['unique_key'].nunique()
print(total_requests)

Top 10 Complaint Categories

In [ ]:
df['complaint_type'].value_counts().head(10)

Agency Workload Distribution

In [24]:
df['agency'].value_counts().head(10)

,count
agency,
NYPD,48335
HPD,22853
DSNY,7242
DOT,5384
DEP,5079
DOB,2497
DHS,2419
DPR,2284
DOHMH,2055


Borough Analysis

In [ ]:
df['borough'].value_counts()

Monthly Request Trend

In [ ]:
df['created_date'] = pd.to_datetime(df['created_date'])

df['month'] = df['created_date'].dt.month_name()

df['month'].value_counts()

Demand by Weekday

In [ ]:
df['weekday'].value_counts()

Demand by Hour

In [ ]:
df['hour'] = df['created_date'].dt.hour
df['hour'].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt #Hours 3, 4, 5, and 6 are missing because there were 0 requests

df['hour'].value_counts().sort_index().plot(
    kind='bar',
    figsize=(10,5)
)

plt.title('Requests by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Requests')
plt.show()

Status Distribution

In [ ]:
df['status'].value_counts()

Resolution_hours calculation

In [ ]:
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

df['resolution_hours'] = (
    df['closed_date'] - df['created_date']
).dt.total_seconds() / 3600

In [ ]:
df['resolution_hours'].describe()

In [25]:
invalid_dates = df[df['resolution_hours'] < 0]

print("Invalid date records:", len(invalid_dates))

Invalid date records: 64


In [ ]:
invalid_dates[
    ['unique_key',
     'created_date',
     'closed_date',
     'resolution_hours']
].head()

In [ ]:
df.loc[
    df['resolution_hours'] < 0,
    'resolution_hours'
] = None


In [ ]:
df['resolution_hours'].describe()

In [ ]:
print(
    "Skewness:",
    df['resolution_hours'].skew()
)

Calculate IQR and outliers

In [ ]:
Q1 = df['resolution_hours'].quantile(0.25)
Q3 = df['resolution_hours'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df['resolution_hours'] < lower) |
    (df['resolution_hours'] > upper)
]

print("Outliers:", len(outliers))

Q1 and Q3 were calculated for resolution_hours.
Using the IQR method (1.5 × IQR), 11,060 records were identified as potential outliers.

These outliers represent requests with unusually long or short resolution durations. According to the project requirements, outliers should be treated as investigation prompts rather than automatically removed, because extended service times may reflect legitimate operational cases.

In [ ]:
print(df[df['resolution_hours'] < 0].shape)

In [ ]:
df.loc[df['resolution_hours'] < 0, 'resolution_hours'] = None

Save the final cleaned datase

In [ ]:
df.to_csv('/content/nyc311_cleaned.csv', index=False)

In [ ]:
df['complaint_type'].value_counts().head(10)

In [ ]:
df['borough'].value_counts()

In [ ]:
df['agency'].value_counts().head(10)

In [ ]:
df.groupby('month').size()

Closure Rate

In [26]:
closure_rate = (
    (df['status'] == 'Closed').sum()
    / len(df)
) * 100

print(f"Closure Rate: {closure_rate:.2f}%")

Closure Rate: 71.38%


Create Status Distribution Chart

In [ ]:
import matplotlib.pyplot as plt

status_counts = df['status'].value_counts()

ax = status_counts.plot(
    kind='bar',
    figsize=(8,5),
    color='skyblue'
)

plt.title('Status Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=45)

for p in ax.patches:
    ax.annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width()/2, p.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()

In [ ]:
df['status'].value_counts()

Top Complaints

In [31]:
top_complaints = df['complaint_type'].value_counts().head(10)
print(top_complaints)


complaint_type
Illegal Parking            17612
Noise - Residential        12144
HEAT/HOT WATER              9083
Noise - Street/Sidewalk     5278
Blocked Driveway            5147
UNSANITARY CONDITION        3721
Noise - Commercial          2284
Noise                       1984
PLUMBING                    1866
Abandoned Vehicle           1748
Name: count, dtype: int64


borough

In [32]:
df['borough'].value_counts()

,count
borough,
BROOKLYN,32235
QUEENS,23159
MANHATTAN,22135
BRONX,18900
STATEN ISLAND,3464
Unspecified,107


agency

In [33]:
df['agency'].value_counts().head(10)

,count
agency,
NYPD,48335
HPD,22853
DSNY,7242
DOT,5384
DEP,5079
DOB,2497
DHS,2419
DPR,2284
DOHMH,2055


Month

In [ ]:
df.groupby('month').size()

In [34]:
print(df['created_date'].min())
print(df['created_date'].max())

2026-10-23 11:49:00
2026-11-04 02:05:23


The extracted dataset contains the most recent 100,000 NYC 311 service requests available through the API. The records span from October 23, 2026 to November 1, 2026. As a result, monthly analysis is concentrated in October (98,460 requests) with a smaller volume in November (1,540 requests). Therefore, monthly trends should be interpreted as a reflection of the extraction window rather than long-term seasonal patterns.

Agency workload distribution

In [35]:
df['agency'].value_counts().head(10)

,count
agency,
NYPD,48335
HPD,22853
DSNY,7242
DOT,5384
DEP,5079
DOB,2497
DHS,2419
DPR,2284
DOHMH,2055


Request volume by borough

In [36]:
df['borough'].value_counts()

,count
borough,
BROOKLYN,32235
QUEENS,23159
MANHATTAN,22135
BRONX,18900
STATEN ISLAND,3464
Unspecified,107


Demand by hour

In [ ]:
df['hour'].value_counts().sort_index()

Median resolution duration by complaint category

In [ ]:
df.groupby('complaint_type')['resolution_hours'].median().sort_values(ascending=False)

Median resolution duration by agency

In [ ]:
df.groupby('agency')['resolution_hours'].median().sort_values(ascending=False)

Geographic complaint concentration

In [ ]:
df['borough'].value_counts()

Recorded closure-rate comparison

In [ ]:
(df['status']=='Closed').mean()*100

Categories with both high request volume and long median resolution duration

In [ ]:
df.groupby('complaint_type').agg({
    'unique_key':'count',
    'resolution_hours':'median'
}).sort_values('unique_key', ascending=False)

A comparison of complaint volume and median resolution duration reveals that HEAT/HOT WATER requests require substantially more time to resolve than most other high-volume complaint categories. This suggests that complaint complexity varies across service categories and should be considered when allocating operational resources.

In [ ]:
df.groupby('agency')['resolution_hours'].median().sort_values(ascending=False)

In [ ]:
((df['status']=='Closed').sum()/len(df))*100

In [37]:
df.to_csv('/content/nyc311_cleaned.csv', index=False)

cleaning code

In [38]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(
    "/content/nyc311_cleaned.csv",
    low_memory=False
)

# Convert date columns
df["created_date"] = pd.to_datetime(
    df["created_date"],
    errors="coerce"
)

df["closed_date"] = pd.to_datetime(
    df["closed_date"],
    errors="coerce"
)

# Fill only descriptive missing values
df["descriptor"] = (
    df["descriptor"]
    .fillna("Not Specified")
    .str.strip()
)

# Recalculate resolution hours
df["resolution_hours"] = (
    df["closed_date"] - df["created_date"]
).dt.total_seconds() / 3600

# Flag invalid date order
df["invalid_date_order"] = (
    df["closed_date"].notna() &
    (df["closed_date"] < df["created_date"])
).astype(int)

# Invalid negative durations must remain missing
df.loc[
    df["invalid_date_order"] == 1,
    "resolution_hours"
] = np.nan

# Closure indicator based on recorded closed_date
df["is_closed"] = df["closed_date"].notna().astype(int)

# Flag missing geographic coordinates
df["missing_coordinates"] = (
    df["latitude"].isna() |
    df["longitude"].isna()
).astype(int)

# Save final version
df.to_csv(
    "/content/nyc311_final_cleaned.csv",
    index=False
)

print("Final cleaned file saved successfully.")

Final cleaned file saved successfully.


In [39]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print(
    "\nInvalid date-order records:",
    df["invalid_date_order"].sum()
)

print(
    "Records missing coordinates:",
    df["missing_coordinates"].sum()
)

print(
    "Duplicate unique keys:",
    df["unique_key"].duplicated().sum()
)

Dataset shape: (100000, 18)

Missing values:
unique_key                 0
created_date               0
closed_date            26297
agency                     0
complaint_type             0
descriptor                 0
borough                    0
status                     0
latitude                1496
longitude               1496
year                       0
month                      0
weekday                    0
hour                       0
is_closed                  0
resolution_hours       26361
invalid_date_order         0
missing_coordinates        0
dtype: int64

Invalid date-order records: 64
Records missing coordinates: 1496
Duplicate unique keys: 0


In [40]:
from google.colab import files

files.download("/content/nyc311_final_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>